### Análise Objetiva de Imagens de Animais com Nvidia Nemotron via OpenRouter

Neste notebook, vamos construir um pipeline simples para analisar imagens de animais usando a API do OpenRouter. Nosso objetivo é extrair uma **descrição puramente objetiva e técnica** da imagem, evitando interpretações emocionais ou análise do cenário.

Para isso, utilizaremos um modelo de Visão da Nvidia (como o Nemotron VL ou NVLM) na sua versão gratuita disponibilizada pelo OpenRouter.

**Passos que abordaremos:**
1. Configuração do ambiente e cliente da API.
2. Teste de conexão simples (apenas texto).
3. Chamada de Visão (Vision) enviando uma imagem em Base64 com nosso prompt objetivo.
---

#### 1. Configuração do Ambiente

Primeiro, precisamos importar as bibliotecas necessárias e carregar nossa chave de API. 
Certifique-se de ter um arquivo `.env` na mesma pasta do seu notebook com a variá

In [ ]:
import os
import base64
from openai import OpenAI
import dotenv

dotenv.load_dotenv()
MODEL_API_KEY = os.getenv("MODEL_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=MODEL_API_KEY,
)

MODEL_ID = "nvidia/nemotron-nano-12b-v2-vl:free"

#### 2. Teste de Conexão (Texto)

Antes de enviarmos imagens pesadas, vamos fazer um teste rápido enviando apenas um prompt de texto. Isso garante que:
- Nossa chave da API está correta.
- A comunicação com o OpenRouter está funcionando.
- O modelo selecionado está respondendo adequadamente.

In [6]:
print("Iniciando teste de conexão...")

try:
    response = client.responses.create(
        model=MODEL_ID,
        input=[
            {"role": "user", "content": "Olá! Responda apenas 'Conexão bem-sucedida' se você estiver recebendo esta mensagem."}
        ]
    )
    
    answer = response.output_text
    print(f"[SUCESSO] Resposta do Modelo: {answer}")

except Exception as e:
    print(f"[ERRO] Falha na conexão: {e}")

Iniciando teste de conexão...
[SUCESSO] Resposta do Modelo: Conexão bem-sucedida



#### 3. Análise Objetiva da Imagem (Vision)

Agora vamos à funcionalidade principal. Modelos de linguagem baseados em visão (VLMs) precisam que a imagem seja enviada em um formato legível para a web, geralmente codificada em **Base64**.

Além disso, nosso prompt deve ser restrito e muito claro para evitar alucinações ou descrições artísticas.

**Nosso Prompt:**
> "Descreva o animal nesta imagem de forma puramente objetiva. Liste apenas: Espécie, Cor predominante, Tamanho estimado, e Marcas ou características físicas distintivas. Não use adjetivos emocionais e não descreva o cenário de fundo"

In [ ]:
# Função auxiliar para converter a imagem local em Base64
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

IMAGE_PATH = "dog.jpg" 
prompt = "Descreva o animal nesta imagem de forma puramente objetiva. Liste apenas: Espécie, Cor predominante, Tamanho estimado, e Marcas ou características físicas distintivas. Não use adjetivos emocionais e não descreva o cenário de fundo"

if os.path.exists(IMAGE_PATH):
    print("Codificando imagem e enviando para o OpenRouter...\n")
    base64_image = encode_image(IMAGE_PATH)
    
    try:
        response = client.responses.create(
            model=MODEL_ID,
            input=[
                {"role": "user",
                 "content": [
                        {"type": "input_text", "text": prompt},
                        {"type": "input_image", "image_url": f"data:image/jpeg;base64,{base64_image}"},
                    ]
                }
            ]
        )
        
        vision_answer = response.output_text
        print("=== DESCRIÇÃO OBJETIVA DO ANIMAL ===")
        print(vision_answer)
        print("======================================")
        
    except Exception as e:
        print(f"[ERRO][ Falha ao processar a imagem: {e}")
        
else:
    print(f"[AVISO][ Imagem não encontrada no caminho: {IMAGE_PATH}. Atualize o caminho da variável IMAGE_PATH para testar.")

Codificando imagem e enviando para o OpenRouter...

=== DESCRIÇÃO OBJETIVA DO ANIMAL ===
Espécie: Golden Retriever  
Cor predominante: Dourado  
Tamanho estimado: Médio a grande  
Marcas ou características físicas distintivas: Pêlo curto a médio dourado com leve claridade no focinho e ponta do rabo; orelhas caídas e alargadas; focinho largo e quadrado; olhos escuros e almôndoa; membros robustos e pêlo que mistura fios levesmente ondulados com o pelo.

